# ApprovalMax CDC Automation Serverless Demo v6

This version separates the CDC demo into layer-specific Unity Catalog schemas and adds ETL audit logging for each step/table.

Layer schemas:
- `workspace.engineer_support_bronze`
- `workspace.engineer_support_silver`
- `workspace.engineer_support_vault`
- `workspace.engineer_support_gold`
- `workspace.engineer_support_quarantine`
- `workspace.engineer_support_monitoring`

Audit log table:
- `workspace.engineer_support_monitoring.etl_audit_log`

The audit log captures:
- run id
- step name
- source layer and target layer
- source table and target table
- status
- row counts
- started/ended timestamps
- duration in seconds
- error message


In [ ]:
from datetime import datetime, timezone, timedelta
import time
import traceback

from pyspark.sql import functions as F
from pyspark.sql.window import Window

catalog = 'workspace'

# Old single-schema demo output. This will be deleted.
old_schema = 'engineer_support'

# Layer-separated schemas.
bronze_schema = 'engineer_support_bronze'
silver_schema = 'engineer_support_silver'
vault_schema = 'engineer_support_vault'
gold_schema = 'engineer_support_gold'
quarantine_schema = 'engineer_support_quarantine'
monitoring_schema = 'engineer_support_monitoring'

DROP_OLD_SCHEMA = True

spark.sql(f'USE CATALOG {catalog}')

if DROP_OLD_SCHEMA:
    spark.sql(f'DROP SCHEMA IF EXISTS {catalog}.{old_schema} CASCADE')
    print(f'Dropped old schema if existed: {catalog}.{old_schema}')

for s in [bronze_schema, silver_schema, vault_schema, gold_schema, quarantine_schema, monitoring_schema]:
    spark.sql(f'CREATE SCHEMA IF NOT EXISTS {catalog}.{s}')
    print(f'Ensured schema: {catalog}.{s}')

def tbl(layer_schema: str, name: str) -> str:
    return f'{catalog}.{layer_schema}.{name}'

def bronze_table(name: str) -> str:
    return tbl(bronze_schema, name)

def silver_table(name: str) -> str:
    return tbl(silver_schema, name)

def vault_table(name: str) -> str:
    return tbl(vault_schema, name)

def gold_table(name: str) -> str:
    return tbl(gold_schema, name)

def quarantine_table(name: str) -> str:
    return tbl(quarantine_schema, name)

def monitoring_table(name: str) -> str:
    return tbl(monitoring_schema, name)

run_id = f'approvalmax_cdc_{datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")}'

audit_table = monitoring_table('etl_audit_log')
spark.sql(f'''
CREATE TABLE IF NOT EXISTS {audit_table} (
  run_id STRING,
  step_name STRING,
  source_layer STRING,
  target_layer STRING,
  source_table STRING,
  target_table STRING,
  status STRING,
  source_row_count BIGINT,
  target_row_count BIGINT,
  failed_row_count BIGINT,
  started_at TIMESTAMP,
  ended_at TIMESTAMP,
  duration_seconds DOUBLE,
  error_message STRING
)
USING DELTA
''')

def _count_table(table_name):
    if not table_name:
        return None
    try:
        return spark.table(table_name).count()
    except Exception:
        return None

def write_audit_log(step_name, source_layer, target_layer, source_table, target_table, status, source_row_count=None, target_row_count=None, failed_row_count=0, started_at=None, ended_at=None, error_message=None):
    started_at = started_at or datetime.now(timezone.utc)
    ended_at = ended_at or datetime.now(timezone.utc)
    duration_seconds = (ended_at - started_at).total_seconds()
    record = [{
        'run_id': run_id,
        'step_name': step_name,
        'source_layer': source_layer,
        'target_layer': target_layer,
        'source_table': source_table,
        'target_table': target_table,
        'status': status,
        'source_row_count': source_row_count,
        'target_row_count': target_row_count,
        'failed_row_count': failed_row_count,
        'started_at': started_at,
        'ended_at': ended_at,
        'duration_seconds': float(duration_seconds),
        'error_message': error_message,
    }]
    spark.createDataFrame(record).write.format('delta').mode('append').saveAsTable(audit_table)

def audited_step(step_name, source_layer, target_layer, source_table, target_table, fn, failed_row_count=0):
    started_at = datetime.now(timezone.utc)
    source_count = _count_table(source_table)
    try:
        result = fn()
        target_count = _count_table(target_table)
        ended_at = datetime.now(timezone.utc)
        write_audit_log(step_name, source_layer, target_layer, source_table, target_table, 'SUCCESS', source_count, target_count, failed_row_count, started_at, ended_at, None)
        print(f'AUDIT SUCCESS | {step_name} | {source_table} -> {target_table} | source={source_count} target={target_count}')
        return result
    except Exception as exc:
        ended_at = datetime.now(timezone.utc)
        write_audit_log(step_name, source_layer, target_layer, source_table, target_table, 'FAILED', source_count, None, failed_row_count, started_at, ended_at, str(exc)[:4000])
        print(f'AUDIT FAILED | {step_name} | {source_table} -> {target_table} | error={exc}')
        raise

print(f'Layered schemas are ready. run_id={run_id}')


In [ ]:
base = datetime.now(timezone.utc).replace(microsecond=0) - timedelta(days=2)

def iso(dt):
    return dt.isoformat().replace('+00:00', 'Z')

def cdc_event(source_table, op, sequence_id, before, after, event_ts, note=None):
    obj = after or before or {}
    pk = {}
    for key in ['company_id', 'document_id', 'event_id', 'workflow_id', 'subscription_id', 'user_id']:
        if obj.get(key) is not None:
            pk[key] = obj[key]
    metadata = {
        'connector': 'mock_debezium_style',
        'database': 'approvalmax_operational',
        'lsn': sequence_id,
        'tx_id': f'TX-{sequence_id//10:06d}',
        'is_snapshot': False,
    }
    if note:
        metadata['note'] = note
    return {
        'schema': 'approvalmax_cdc_v1',
        'source_system': 'approvalmax_mock_cdc',
        'source_table': source_table,
        'op': op,
        'sequence_id': sequence_id,
        'event_timestamp': iso(event_ts),
        'ingestion_timestamp': iso(event_ts + timedelta(seconds=20)),
        'primary_key': pk,
        'before': before,
        'after': after,
        'metadata': metadata,
    }

company = {
    'company_id': 'COMP-101',
    'company_name': 'Demo Finance Ltd',
    'country': 'United Kingdom',
    'base_currency': 'GBP',
    'status': 'active',
    'created_at': iso(base - timedelta(days=120)),
    'updated_at': iso(base),
}

po_draft = {
    'document_id': 'PO-CDC-10001',
    'company_id': 'COMP-101',
    'document_type': 'purchase_order',
    'document_status': 'draft',
    'supplier_id': 'SUP-501',
    'supplier_name': 'London IT Services',
    'total_amount': 1250.75,
    'currency': 'GBP',
    'created_at': iso(base + timedelta(minutes=5)),
    'submitted_at': None,
    'approved_at': None,
    'rejected_at': None,
    'paid_at': None,
    'updated_at': iso(base + timedelta(minutes=5)),
    'approval_deadline_at': None,
}

po_submitted = dict(po_draft)
po_submitted.update({
    'document_status': 'submitted',
    'submitted_at': iso(base + timedelta(hours=1)),
    'updated_at': iso(base + timedelta(hours=1)),
})

po_approved = dict(po_submitted)
po_approved.update({
    'document_status': 'approved',
    'approved_at': iso(base + timedelta(hours=5)),
    'approval_deadline_at': iso(base + timedelta(hours=8)),
    'updated_at': iso(base + timedelta(hours=5)),
})

submitted_event = {
    'event_id': 'EVT-CDC-10001',
    'document_id': 'PO-CDC-10001',
    'workflow_id': 'WF-COMP-101-AP-01',
    'company_id': 'COMP-101',
    'approver_user_id': 'USR-101-01',
    'event_type': 'submitted',
    'event_timestamp': iso(base + timedelta(hours=1)),
}

approved_event = {
    'event_id': 'EVT-CDC-10002',
    'document_id': 'PO-CDC-10001',
    'workflow_id': 'WF-COMP-101-AP-01',
    'company_id': 'COMP-101',
    'approver_user_id': 'USR-101-02',
    'event_type': 'approved',
    'event_timestamp': iso(base + timedelta(hours=5)),
}

records = [
    cdc_event('companies', 'c', 200001, None, company, base),
    cdc_event('finance_documents', 'c', 200002, None, po_draft, base + timedelta(minutes=5)),
    cdc_event('finance_documents', 'u', 200003, po_draft, po_submitted, base + timedelta(hours=1)),
    cdc_event('approval_events', 'c', 200004, None, submitted_event, base + timedelta(hours=1, minutes=1)),
    cdc_event('finance_documents', 'u', 200005, po_submitted, po_approved, base + timedelta(hours=5), 'additive_schema_evolution_approval_deadline_at'),
    cdc_event('approval_events', 'c', 200006, None, approved_event, base + timedelta(hours=5, minutes=1)),
]

bronze_df = spark.createDataFrame(records)
bronze_df = bronze_df.withColumn('_loaded_at', F.current_timestamp())
def write_bronze():
    bronze_df.write.format('delta').mode('overwrite').option('overwriteSchema', 'true').saveAsTable(bronze_table('approvalmax_cdc_raw'))

audited_step(
    step_name='bronze_load_approvalmax_cdc_raw',
    source_layer='source',
    target_layer='bronze',
    source_table='inline_mock_cdc_records',
    target_table=bronze_table('approvalmax_cdc_raw'),
    fn=write_bronze,
)
print(f'Loaded {bronze_df.count()} CDC rows into {bronze_table("approvalmax_cdc_raw")}')


In [ ]:
bronze = spark.table(bronze_table('approvalmax_cdc_raw'))

def latest_by_key(df, key_col, ts_col='event_timestamp'):
    order_cols = [F.col('sequence_id').desc_nulls_last()]
    if ts_col in df.columns:
        order_cols.append(F.col(ts_col).desc_nulls_last())
    w = Window.partitionBy(key_col).orderBy(*order_cols)
    return df.withColumn('_rn', F.row_number().over(w)).filter(F.col('_rn') == 1).drop('_rn')

companies = bronze.filter((F.col('source_table') == 'companies') & (F.col('op') != 'd')).select(
    F.col('after.company_id').alias('company_id'),
    F.col('after.company_name').alias('company_name'),
    F.col('after.country').alias('country'),
    F.col('after.base_currency').alias('base_currency'),
    F.col('after.status').alias('company_status'),
    F.col('after.created_at').cast('timestamp').alias('created_at'),
    F.col('after.updated_at').cast('timestamp').alias('updated_at'),
    F.col('sequence_id'),
    F.col('event_timestamp').cast('timestamp').alias('event_timestamp'),
    F.col('ingestion_timestamp').cast('timestamp').alias('ingestion_timestamp'),
    F.col('source_system').alias('record_source'),
)
audited_step('bronze_to_silver_companies_current', 'bronze', 'silver', bronze_table('approvalmax_cdc_raw'), silver_table('companies_current'), lambda: latest_by_key(companies, 'company_id', 'ingestion_timestamp').write.format('delta').mode('overwrite').option('overwriteSchema', 'true').saveAsTable(silver_table('companies_current')))

docs = bronze.filter((F.col('source_table') == 'finance_documents') & (F.col('op') != 'd')).select(
    F.col('after.document_id').alias('document_id'),
    F.col('after.company_id').alias('company_id'),
    F.col('after.document_type').alias('document_type'),
    F.col('after.document_status').alias('document_status'),
    F.col('after.supplier_id').alias('supplier_id'),
    F.col('after.supplier_name').alias('supplier_name'),
    F.col('after.total_amount').cast('decimal(18,2)').alias('total_amount'),
    F.col('after.currency').alias('currency'),
    F.col('after.created_at').cast('timestamp').alias('created_at'),
    F.col('after.submitted_at').cast('timestamp').alias('submitted_at'),
    F.col('after.approved_at').cast('timestamp').alias('approved_at'),
    F.col('after.rejected_at').cast('timestamp').alias('rejected_at'),
    F.col('after.paid_at').cast('timestamp').alias('paid_at'),
    F.col('after.updated_at').cast('timestamp').alias('updated_at'),
    F.col('after.approval_deadline_at').cast('timestamp').alias('approval_deadline_at'),
    F.col('sequence_id'),
    F.col('event_timestamp').cast('timestamp').alias('event_timestamp'),
    F.col('ingestion_timestamp').cast('timestamp').alias('ingestion_timestamp'),
    F.col('source_system').alias('record_source'),
)
audited_step('bronze_to_silver_finance_documents_current', 'bronze', 'silver', bronze_table('approvalmax_cdc_raw'), silver_table('finance_documents_current'), lambda: latest_by_key(docs, 'document_id', 'ingestion_timestamp').write.format('delta').mode('overwrite').option('overwriteSchema', 'true').saveAsTable(silver_table('finance_documents_current')))

events = bronze.filter((F.col('source_table') == 'approval_events') & (F.col('op') != 'd')).select(
    F.col('after.event_id').alias('event_id'),
    F.col('after.document_id').alias('document_id'),
    F.col('after.workflow_id').alias('workflow_id'),
    F.col('after.company_id').alias('company_id'),
    F.col('after.approver_user_id').alias('approver_user_id'),
    F.col('after.event_type').alias('event_type'),
    F.col('after.event_timestamp').cast('timestamp').alias('approval_event_timestamp'),
    F.col('sequence_id'),
    F.col('event_timestamp').cast('timestamp').alias('cdc_event_timestamp'),
    F.col('ingestion_timestamp').cast('timestamp').alias('ingestion_timestamp'),
    F.col('source_system').alias('record_source'),
)
audited_step('bronze_to_silver_approval_events', 'bronze', 'silver', bronze_table('approvalmax_cdc_raw'), silver_table('approval_events'), lambda: latest_by_key(events, 'event_id', 'ingestion_timestamp').write.format('delta').mode('overwrite').option('overwriteSchema', 'true').saveAsTable(silver_table('approval_events')))
print('Built Silver tables')


In [ ]:

vault_builds = [
    ('silver_to_vault_hub_company', silver_table('companies_current'), vault_table('hub_company'),
     f'''CREATE OR REPLACE TABLE {vault_table('hub_company')} AS SELECT DISTINCT sha2(company_id, 256) AS company_hk, company_id AS company_bk, current_timestamp() AS load_datetime, record_source FROM {silver_table('companies_current')} WHERE company_id IS NOT NULL'''),
    ('silver_to_vault_hub_finance_document', silver_table('finance_documents_current'), vault_table('hub_finance_document'),
     f'''CREATE OR REPLACE TABLE {vault_table('hub_finance_document')} AS SELECT DISTINCT sha2(document_id, 256) AS finance_document_hk, document_id AS finance_document_bk, current_timestamp() AS load_datetime, record_source FROM {silver_table('finance_documents_current')} WHERE document_id IS NOT NULL'''),
    ('silver_to_vault_hub_approval_event', silver_table('approval_events'), vault_table('hub_approval_event'),
     f'''CREATE OR REPLACE TABLE {vault_table('hub_approval_event')} AS SELECT DISTINCT sha2(event_id, 256) AS approval_event_hk, event_id AS approval_event_bk, current_timestamp() AS load_datetime, record_source FROM {silver_table('approval_events')} WHERE event_id IS NOT NULL'''),
    ('silver_to_vault_link_document_company', silver_table('finance_documents_current'), vault_table('link_document_company'),
     f'''CREATE OR REPLACE TABLE {vault_table('link_document_company')} AS SELECT DISTINCT sha2(concat_ws('||', document_id, company_id), 256) AS document_company_hk, sha2(document_id, 256) AS finance_document_hk, sha2(company_id, 256) AS company_hk, current_timestamp() AS load_datetime, record_source FROM {silver_table('finance_documents_current')} WHERE document_id IS NOT NULL AND company_id IS NOT NULL'''),
    ('silver_to_vault_link_document_approval_event', silver_table('approval_events'), vault_table('link_document_approval_event'),
     f'''CREATE OR REPLACE TABLE {vault_table('link_document_approval_event')} AS SELECT DISTINCT sha2(concat_ws('||', document_id, event_id), 256) AS document_approval_event_hk, sha2(document_id, 256) AS finance_document_hk, sha2(event_id, 256) AS approval_event_hk, current_timestamp() AS load_datetime, record_source FROM {silver_table('approval_events')} WHERE document_id IS NOT NULL AND event_id IS NOT NULL'''),
    ('silver_to_vault_sat_finance_document_status', silver_table('finance_documents_current'), vault_table('sat_finance_document_status'),
     f'''CREATE OR REPLACE TABLE {vault_table('sat_finance_document_status')} AS SELECT sha2(document_id, 256) AS finance_document_hk, document_status, document_type, supplier_id, supplier_name, created_at, submitted_at, approved_at, rejected_at, paid_at, updated_at, approval_deadline_at, sha2(concat_ws('||', coalesce(document_status, ''), coalesce(document_type, ''), coalesce(cast(updated_at as string), ''), coalesce(cast(approval_deadline_at as string), '')), 256) AS hashdiff, current_timestamp() AS load_datetime, record_source FROM {silver_table('finance_documents_current')} WHERE document_id IS NOT NULL'''),
    ('silver_to_vault_sat_finance_document_amount', silver_table('finance_documents_current'), vault_table('sat_finance_document_amount'),
     f'''CREATE OR REPLACE TABLE {vault_table('sat_finance_document_amount')} AS SELECT sha2(document_id, 256) AS finance_document_hk, total_amount, currency, sha2(concat_ws('||', coalesce(cast(total_amount as string), ''), coalesce(currency, '')), 256) AS hashdiff, current_timestamp() AS load_datetime, record_source FROM {silver_table('finance_documents_current')} WHERE document_id IS NOT NULL'''),
    ('silver_to_vault_sat_approval_event_detail', silver_table('approval_events'), vault_table('sat_approval_event_detail'),
     f'''CREATE OR REPLACE TABLE {vault_table('sat_approval_event_detail')} AS SELECT sha2(event_id, 256) AS approval_event_hk, workflow_id, document_id, company_id, approver_user_id, event_type, approval_event_timestamp, sha2(concat_ws('||', coalesce(workflow_id, ''), coalesce(document_id, ''), coalesce(company_id, ''), coalesce(approver_user_id, ''), coalesce(event_type, ''), coalesce(cast(approval_event_timestamp as string), '')), 256) AS hashdiff, current_timestamp() AS load_datetime, record_source FROM {silver_table('approval_events')} WHERE event_id IS NOT NULL'''),
]

for step_name, source_table, target_table, sql_statement in vault_builds:
    audited_step(
        step_name=step_name,
        source_layer='silver',
        target_layer='vault',
        source_table=source_table,
        target_table=target_table,
        fn=lambda s=sql_statement: spark.sql(s)
    )

print('Built Raw Vault-style tables')


In [ ]:

gold_sql = f'''
CREATE OR REPLACE TABLE {gold_table('fact_approval_document_lifecycle')} AS
SELECT d.document_id, d.company_id, c.company_name, d.document_type, d.document_status, d.supplier_id, d.supplier_name, d.total_amount, d.currency, d.created_at, d.submitted_at, d.approved_at, d.rejected_at, d.paid_at, d.updated_at, d.approval_deadline_at,
       CASE WHEN d.submitted_at IS NOT NULL AND d.approved_at IS NOT NULL THEN timestampdiff(MINUTE, d.submitted_at, d.approved_at) END AS approval_cycle_time_minutes,
       CASE WHEN d.approval_deadline_at IS NOT NULL AND d.approved_at IS NOT NULL AND d.approved_at > d.approval_deadline_at THEN 1 ELSE 0 END AS approval_sla_breach_flag
FROM {silver_table('finance_documents_current')} d
LEFT JOIN {silver_table('companies_current')} c ON d.company_id = c.company_id
'''

audited_step(
    step_name='silver_to_gold_fact_approval_document_lifecycle',
    source_layer='silver',
    target_layer='gold',
    source_table=silver_table('finance_documents_current'),
    target_table=gold_table('fact_approval_document_lifecycle'),
    fn=lambda: spark.sql(gold_sql)
)
print('Built Gold mart')


In [ ]:

checks = {
    'document_id_not_null': f'SELECT * FROM {silver_table("finance_documents_current")} WHERE document_id IS NULL',
    'company_id_not_null': f'SELECT * FROM {silver_table("finance_documents_current")} WHERE company_id IS NULL',
    'document_status_accepted': f"SELECT * FROM {silver_table('finance_documents_current')} WHERE document_status NOT IN ('draft', 'submitted', 'approved', 'rejected', 'paid', 'cancelled') OR document_status IS NULL",
    'total_amount_non_negative': f'SELECT * FROM {silver_table("finance_documents_current")} WHERE total_amount < 0',
    'approval_event_timestamp_not_future': f'SELECT * FROM {silver_table("approval_events")} WHERE approval_event_timestamp > current_timestamp()',
    'gold_approval_cycle_time_non_negative': f'SELECT * FROM {gold_table("fact_approval_document_lifecycle")} WHERE approval_cycle_time_minutes < 0',
    'gold_grain_unique': f'SELECT document_id, COUNT(*) AS row_count FROM {gold_table("fact_approval_document_lifecycle")} GROUP BY document_id HAVING COUNT(*) > 1',
}

failures = []
for name, query in checks.items():
    started_at = datetime.now(timezone.utc)
    failed = spark.sql(query)
    count = failed.count()
    qtable = quarantine_table(f'{name}')
    print(f'{name}: {count} failing rows')
    if count > 0:
        failed.write.format('delta').mode('overwrite').option('overwriteSchema', 'true').saveAsTable(qtable)
        failures.append((name, count, qtable))
        write_audit_log(
            step_name=f'quality_check_{name}',
            source_layer='quality',
            target_layer='quarantine',
            source_table='query_based_quality_check',
            target_table=qtable,
            status='FAILED',
            source_row_count=None,
            target_row_count=count,
            failed_row_count=count,
            started_at=started_at,
            ended_at=datetime.now(timezone.utc),
            error_message=f'{count} failing rows written to {qtable}'
        )
    else:
        write_audit_log(
            step_name=f'quality_check_{name}',
            source_layer='quality',
            target_layer='quality',
            source_table='query_based_quality_check',
            target_table=None,
            status='SUCCESS',
            source_row_count=None,
            target_row_count=0,
            failed_row_count=0,
            started_at=started_at,
            ended_at=datetime.now(timezone.utc),
            error_message=None
        )

if failures:
    raise Exception('Quality checks failed: ' + str(failures))

display(spark.table(gold_table('fact_approval_document_lifecycle')))
display(spark.sql(f"SELECT * FROM {audit_table} WHERE run_id = '{run_id}' ORDER BY started_at"))
print('All quality checks passed')
